# Shared Preprocessing 

**ALL FOUR MODELS MUST IMPORT FROM THIS NOTEBOOK.** This guarantees identical train/val/test splits, normalization statistics, missing-value handling and class-imbalance strategy across all four architectures, so the final comparison in Section 7 of the report is fair (same preprocessing = only the architecture differs).

**How to use this from your own model notebook** (see the last section of this notebook for full instructions).


## 1.Fixed for every member

In [5]:
import glob
import json
import os
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import tensorflow as tf
import torch
from torch.utils.data import Dataset, DataLoader

SEED = 42
PATCH_SIZE = 64

INPUT_FEATURES: List[str] = [
    "elevation", "th", "vs", "tmmn", "tmmx", "sph", "pr",
    "pdsi", "NDVI", "population", "erc", "PrevFireMask",
]
TARGET_FEATURE = "FireMask"
ALL_FEATURES = INPUT_FEATURES + [TARGET_FEATURE]
NUM_CHANNELS = len(INPUT_FEATURES)

UNCERTAIN_LABEL_VALUE = -1.0
WIND_DIRECTION_CHANNEL = INPUT_FEATURES.index("th")

# Physical ranges used to identify corrupted/sentinel input values in the EDA.
# Bounds are applied using training-derived medians, then reused for val/test.
FEATURE_VALID_RANGES = {
    "elevation": (-500.0, 9000.0),
    "th": (0.0, 360.0),
    "vs": (0.0, 30.0),
    "tmmn": (200.0, 330.0),
    "tmmx": (200.0, 350.0),
    "sph": (0.0, 0.03),
    "pr": (0.0, 50.0),
    "pdsi": (-10.0, 10.0),
    "NDVI": (-10000.0, 10000.0),
    "population": (0.0, 100000.0),
    "erc": (0.0, 120.0),
    "PrevFireMask": (-1.0, 1.0),
}

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
STATS_PATH = str(PROJECT_ROOT / "shared" / "normalization_stats.json")
DEFAULT_DATA_DIR = str(DATA_DIR)

np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Raw TFRecord parsing

Reads the dataset's `.tfrecord` files and parses each example into a 12-channel input stack + 1-channel target mask.

In [6]:
def _feature_description() -> Dict[str, "tf.io.FixedLenFeature"]:
    return {
        name: tf.io.FixedLenFeature([PATCH_SIZE, PATCH_SIZE], tf.float32)
        for name in ALL_FEATURES
    }


def _parse_example(example_proto):
    parsed = tf.io.parse_single_example(example_proto, _feature_description())
    inputs = tf.stack([parsed[name] for name in INPUT_FEATURES], axis=0)
    target = parsed[TARGET_FEATURE][tf.newaxis, ...]
    return inputs, target


def load_split_arrays(tfrecord_paths: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    """Read TFRecords into inputs (N, 12, 64, 64) and targets (N, 1, 64, 64)."""
    if not tfrecord_paths:
        raise FileNotFoundError(
            "No TFRecord files provided. Put the dataset files directly in data/."
        )

    raw_dataset = tf.data.TFRecordDataset(tfrecord_paths)
    parsed_dataset = raw_dataset.map(_parse_example)

    all_inputs, all_targets = [], []
    for inputs, target in parsed_dataset:
        all_inputs.append(inputs.numpy())
        all_targets.append(target.numpy())

    if not all_inputs:
        raise ValueError(f"TFRecord files were found but contained 0 examples: {tfrecord_paths}")

    return np.stack(all_inputs), np.stack(all_targets)

## 3. Fixed train / val / test split

Uses the dataset's **own predefined** train/eval/test TFRecord files rather than re-shuffling everything ourselves. This avoids leaking spatially/temporally correlated patches (e.g. the same fire on adjacent days) across splits, and guarantees all 4 members get an identical split by construction.

In [7]:
def get_split_files(data_dir: str) -> Dict[str, List[str]]:
    splits = {
        "train": sorted(glob.glob(os.path.join(data_dir, "*train*.tfrecord*"))),
        "val": sorted(glob.glob(os.path.join(data_dir, "*eval*.tfrecord*"))),
        "test": sorted(glob.glob(os.path.join(data_dir, "*test*.tfrecord*"))),
    }
    for split_name, files in splits.items():
        if not files:
            raise FileNotFoundError(
                f"No files found for split '{split_name}' in '{data_dir}'. "
                "Check that the downloaded filenames contain 'train' / 'eval' / "
                "'test', or adjust the glob pattern in get_split_files()."
            )
    return splits


## 4. Missing values and invalid sensor values

The EDA found no NaNs in the training split, but it did find physically invalid/extreme input values. For example, wind direction should be in `[0, 360]`, temperatures should be plausible Kelvin values, and precipitation/speed/humidity should not be negative.

The shared pipeline replaces values outside `FEATURE_VALID_RANGES` with the median of valid **training** values for that channel. The same training medians and bounds are reused for validation and test data. This prevents outliers from corrupting normalization and prevents validation/test information from leaking into preprocessing.

The target uses `-1` for uncertain/unlabeled pixels. Every model must exclude those pixels using `valid_mask`.

In [8]:
def build_valid_mask(targets: np.ndarray) -> np.ndarray:
    """1.0 where the label is 0/1, 0.0 where it is the -1 sentinel."""
    return (targets != UNCERTAIN_LABEL_VALUE).astype(np.float32)


def _feature_valid_mask(values: np.ndarray, feature_name: str) -> np.ndarray:
    lower, upper = FEATURE_VALID_RANGES[feature_name]
    return np.isfinite(values) & (values >= lower) & (values <= upper)


def _clean_inputs_with_medians(
    inputs: np.ndarray,
    channel_medians: np.ndarray,
) -> np.ndarray:
    """Replace NaN and EDA-identified invalid values with train medians.

    The medians must be computed from valid training values only and reused for
    validation/test data. This prevents invalid raw values and split leakage
    from distorting normalization.
    """
    cleaned = inputs.copy()
    for channel_index, feature_name in enumerate(INPUT_FEATURES):
        values = cleaned[:, channel_index, :, :]
        valid = _feature_valid_mask(values, feature_name)
        values[~valid] = channel_medians[channel_index]
        cleaned[:, channel_index, :, :] = values
    return cleaned


def _fill_nans_with_medians(inputs: np.ndarray, channel_medians: np.ndarray) -> np.ndarray:
    """Backward-compatible wrapper for input cleaning."""
    return _clean_inputs_with_medians(inputs, channel_medians)

## 5. Normalization (train-set statistics only, saved + shared)

Per-channel z-score normalization. Stats are computed **once**, on the training set only, saved to `normalization_stats.json`, and reused by every member — this satisfies the assignment requirement that preprocessing decisions use only train/validation data.

In [9]:
@dataclass
class NormalizationStats:
    means: List[float]
    stds: List[float]
    medians: List[float]
    version: int = 2

    def save(self, path: str = STATS_PATH) -> None:
        with open(path, "w") as file:
            json.dump(asdict(self), file, indent=2)

    @classmethod
    def load(cls, path: str = STATS_PATH) -> "NormalizationStats":
        with open(path) as file:
            payload = json.load(file)
        payload.setdefault("version", 1)
        return cls(**payload)


def compute_normalization_stats(train_inputs: np.ndarray) -> NormalizationStats:
    means, stds, medians = [], [], []
    for channel_index, feature_name in enumerate(INPUT_FEATURES):
        channel = train_inputs[:, channel_index, :, :]
        lower, upper = FEATURE_VALID_RANGES[feature_name]
        valid_values = channel[
            np.isfinite(channel) & (channel >= lower) & (channel <= upper)
        ]
        if valid_values.size == 0:
            raise ValueError(f"No valid training values found for feature '{feature_name}'.")
        median = float(np.median(valid_values))
        clean_channel = np.where(
            _feature_valid_mask(channel, feature_name), channel, median
        )
        medians.append(median)
        means.append(float(clean_channel.mean()))
        stds.append(float(clean_channel.std() + 1e-8))
    return NormalizationStats(means=means, stds=stds, medians=medians)


def normalize_inputs(inputs: np.ndarray, stats: NormalizationStats) -> np.ndarray:
    inputs = _clean_inputs_with_medians(inputs, np.array(stats.medians))
    means = np.array(stats.means).reshape(1, -1, 1, 1)
    stds = np.array(stats.stds).reshape(1, -1, 1, 1)
    return (inputs - means) / stds

## 6. Class-imbalance strategy

Fire pixels are heavily outnumbered by no-fire pixels. `pos_weight` is computed once on the training set and must be used identically by every member's loss function (e.g. `torch.nn.BCEWithLogitsLoss(pos_weight=...)`), so the imbalance strategy is a controlled variable, not a hidden confound.

In [10]:
def compute_pos_weight(train_targets: np.ndarray) -> float:
    valid_mask = build_valid_mask(train_targets).astype(bool)
    valid_labels = train_targets[valid_mask]
    num_pos = (valid_labels == 1).sum()
    num_neg = (valid_labels == 0).sum()
    if num_pos == 0:
        raise ValueError("No positive (fire) pixels found in the training targets — check parsing.")
    return float(num_neg / num_pos)


## 7. PyTorch Dataset

Wraps the numpy arrays into a `Dataset` that returns `(input, target, valid_mask)` triples. Includes augmentation with a fix most implementations miss: **wind direction (`th`) is directional**, in degrees — flipping/rotating the grid without transforming the wind-direction *values* would create physically inconsistent samples.

In [11]:
class WildfireDataset(Dataset):
    """Return normalized input, target, and valid-mask tensors."""

    def __init__(
        self,
        inputs: np.ndarray,
        targets: np.ndarray,
        augment: bool = False,
        wind_direction_mean: float = 0.0,
        wind_direction_std: float = 1.0,
    ):
        self.inputs = inputs
        self.targets = targets
        self.valid_mask = build_valid_mask(targets)
        self.augment = augment
        self.wind_direction_mean = wind_direction_mean
        self.wind_direction_std = wind_direction_std

    def __len__(self) -> int:
        return len(self.inputs)

    def __getitem__(self, idx: int):
        x = self.inputs[idx].copy()
        y = self.targets[idx].copy()
        m = self.valid_mask[idx].copy()

        if self.augment:
            x, y, m = self._augment(x, y, m)

        return (
            torch.from_numpy(np.ascontiguousarray(x)).float(),
            torch.from_numpy(np.ascontiguousarray(y)).float(),
            torch.from_numpy(np.ascontiguousarray(m)).float(),
        )

    def _augment(self, x: np.ndarray, y: np.ndarray, m: np.ndarray):
        # Convert normalized wind direction back to degrees before geometric transforms.
        x[WIND_DIRECTION_CHANNEL] = (
            x[WIND_DIRECTION_CHANNEL] * self.wind_direction_std
            + self.wind_direction_mean
        )

        if np.random.rand() < 0.5:
            x = x[:, :, ::-1]
            y = y[:, :, ::-1]
            m = m[:, :, ::-1]
            x[WIND_DIRECTION_CHANNEL] = (180.0 - x[WIND_DIRECTION_CHANNEL]) % 360.0

        if np.random.rand() < 0.5:
            x = x[:, ::-1, :]
            y = y[:, ::-1, :]
            m = m[:, ::-1, :]
            x[WIND_DIRECTION_CHANNEL] = (360.0 - x[WIND_DIRECTION_CHANNEL]) % 360.0

        k = np.random.randint(0, 4)
        if k:
            x = np.rot90(x, k=k, axes=(1, 2))
            y = np.rot90(y, k=k, axes=(1, 2))
            m = np.rot90(m, k=k, axes=(1, 2))
            x[WIND_DIRECTION_CHANNEL] = (x[WIND_DIRECTION_CHANNEL] - 90.0 * k) % 360.0

        x[WIND_DIRECTION_CHANNEL] = (
            x[WIND_DIRECTION_CHANNEL] - self.wind_direction_mean
        ) / self.wind_direction_std
        return x, y, m

## 8. Top-level entry point

**This is the only function other notebooks should call.**

In [12]:
def get_dataloaders(
    data_dir: str = DEFAULT_DATA_DIR,
    batch_size: int = 32,
    augment_train: bool = True,
    num_workers: int = 0,
) -> Tuple[DataLoader, DataLoader, DataLoader, float]:
    """Return shared train, validation, test loaders and positive-class weight."""
    split_files = get_split_files(data_dir)

    train_inputs, train_targets = load_split_arrays(split_files["train"])
    val_inputs, val_targets = load_split_arrays(split_files["val"])
    test_inputs, test_targets = load_split_arrays(split_files["test"])

    stats = None
    if os.path.exists(STATS_PATH):
        candidate = NormalizationStats.load(STATS_PATH)
        if candidate.version == 2:
            stats = candidate

    if stats is None:
        stats = compute_normalization_stats(train_inputs)
        stats.save(STATS_PATH)

    train_inputs = normalize_inputs(train_inputs, stats)
    val_inputs = normalize_inputs(val_inputs, stats)
    test_inputs = normalize_inputs(test_inputs, stats)
    pos_weight = compute_pos_weight(train_targets)

    wind_mean = stats.means[WIND_DIRECTION_CHANNEL]
    wind_std = stats.stds[WIND_DIRECTION_CHANNEL]
    train_ds = WildfireDataset(
        train_inputs,
        train_targets,
        augment=augment_train,
        wind_direction_mean=wind_mean,
        wind_direction_std=wind_std,
    )
    val_ds = WildfireDataset(val_inputs, val_targets)
    test_ds = WildfireDataset(test_inputs, test_targets)

    generator = torch.Generator()
    generator.manual_seed(SEED)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        generator=generator,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    return train_loader, val_loader, test_loader, pos_weight

## 9. Quick self-test

Run `run_preprocessing_smoke_test()` manually after downloading the dataset to `data/` to confirm everything loads correctly.

In [14]:
def run_preprocessing_smoke_test():
    """Load one shared batch and verify the agreed model input contract."""
    train_loader, val_loader, test_loader, pos_weight = get_dataloaders()

    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches:   {len(val_loader)}")
    print(f"Test batches:  {len(test_loader)}")
    print(f"pos_weight for BCEWithLogitsLoss: {pos_weight:.3f}")

    x, y, m = next(iter(train_loader))
    print(f"Input batch shape:  {tuple(x.shape)}")
    print(f"Target batch shape: {tuple(y.shape)}")
    print(f"Mask batch shape:   {tuple(m.shape)}")

    assert x.shape[1:] == (NUM_CHANNELS, PATCH_SIZE, PATCH_SIZE)
    assert y.shape[1:] == (1, PATCH_SIZE, PATCH_SIZE)
    assert m.shape == y.shape
    assert torch.isfinite(x).all()
    print("Preprocessing smoke test passed.")